# 📊 Notebook 09: Model Evaluation & Comparison

> Phase 4 of the FIFA World Cup Predictor: Comprehensive evaluation of all baseline and advanced gradient-boosted models (Logistic Regression, Random Forest, Gradient Boosting, XGBoost, LightGBM) on modern unseen fixtures (2021–2026).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add src to Python path
sys.path.insert(0, '../src')

from evaluate_model import evaluate_saved_model
from train_model import LABEL_MAP


## 1. Execute Multi-Model Temporal Benchmark

Evaluate all saved model checkpoints in `models/` on matches from 2021–2026 (5,779 test matches).


In [ ]:
eval_summary = evaluate_saved_model(split_strategy='temporal')

comparison_rows = []
for name, m in eval_summary['models'].items():
    comparison_rows.append({
        'Model': name,
        'Accuracy': f"{m['accuracy']:.2%}",
        'Balanced Accuracy': f"{m['balanced_accuracy']:.2%}",
        'Log Loss': f"{m['log_loss']:.4f}",
        'Macro F1': f"{m['macro_f1']:.4f}"
    })

comp_df = pd.DataFrame(comparison_rows).sort_values('Accuracy', ascending=False)
print("\n--- Model Leaderboard ---")
display(comp_df)


## 2. Confusion Matrix Inspection

Examine actual vs. predicted outcomes for the leading models.


In [ ]:
labels = [LABEL_MAP[i] for i in range(3)]
for name, m in eval_summary['models'].items():
    if name in ['best_model', 'xgboost', 'lightgbm']:
        cm = np.array(m['confusion_matrix']['rows_actual_columns_predicted'])
        print(f"\nConfusion Matrix: {name}")
        cm_df = pd.DataFrame(cm, index=[f'Actual {l}' for l in labels], columns=[f'Pred {l}' for l in labels])
        display(cm_df)


## 3. Class-by-Class Classification Metrics


In [ ]:
best_report = eval_summary['models']['best_model']['classification_report']
print("Best Model Classification Report:")
for label in labels:
    metrics = best_report[label]
    print(f"  {label:10} | Precision: {metrics['precision']:.2%} | Recall: {metrics['recall']:.2%} | F1: {metrics['f1-score']:.4f} | Support: {int(metrics['support'])}")
